# 04 — Model Training

## Purpose

Train and compare five models on the EU-27 GDP growth dataset:

1. **Naive baseline — mean**: predicts the training mean for every test row.
2. **Naive baseline — persistence**: predicts next year's growth = this year's growth.
3. **Linear Regression**: simple, interpretable, OLS baseline.
4. **Random Forest**: tree-based ensemble reference model.
5. **XGBoost**: the flagship model for this project.

## Methodological notes

- **Data**: 810 country-year observations, EU-27, predictor years 1995–2024
  (target years 1996–2025).
- **Target**: next-year GDP growth (`gdp_growth` shifted −1 within each
  country). The `gdp_growth` column in the features is the *current* year's
  value — the target is a separate column and is excluded from X.
- **Split**: *chronological* (train predictors 1995–2019, test predictors
  2020–2024). No shuffling — this avoids look-ahead bias.
- Each model is wrapped in a scikit-learn `Pipeline` with median imputation
  and standard scaling, fit only on training data to prevent leakage.
- **Random seed**: `SEED = 8` in `src/config.py`, aliased to `RANDOM_STATE`.
  All models use this seed for reproducibility.
- All logic lives in `src/` modules; this notebook imports and orchestrates it.

In [9]:
import sys
from pathlib import Path

current = Path.cwd()
project_root = current.parent if current.name == "notebooks" else current
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

In [10]:
import pandas as pd
import numpy as np
from src import config, split, models, evaluate

In [11]:
df = pd.read_csv(config.MODEL_DATA_PATH)
print("Shape:", df.shape)
print("Split counts:")
print(df[config.SPLIT_COL].value_counts())

Shape: (810, 19)
Split counts:
dataset_split
train    675
test     135
Name: count, dtype: int64


In [12]:
X_train, y_train, X_test, y_test = split.get_train_test(df)
print("Train:", X_train.shape, " Test:", X_test.shape)
print("Features:", X_train.columns.tolist())

Train: (675, 11)  Test: (135, 11)
Features: ['gdp_growth', 'gdp_per_capita', 'inflation', 'gross_fixed_capital_formation', 'unemployment', 'trade_openness', 'fdi_inflows_percent_gdp', 'government_consumption_percent_gdp', 'population_growth', 'gdp_growth_lag_1', 'gdp_growth_lag_2']


## Model Comparison

We evaluate five models on the same chronological train/test split.
For each model we report:

- **MAE** (primary metric): average absolute error in GDP-growth percentage points.
- **RMSE**: same units as MAE but penalises large errors more strongly.
- **R²**: proportion of variance explained; **negative values mean the model
  is worse than predicting the mean** of the test set.

In [13]:
model_candidates = [
    ("Naive: mean",         models.build_mean_baseline()),
    ("Naive: persistence",  models.build_persistence_baseline()),
    ("Linear Regression",   models.build_linear_regression()),
    ("Random Forest",       models.build_random_forest()),
    ("XGBoost (baseline)",  models.build_xgboost_baseline()),
]

results = evaluate.build_comparison_table(
    model_candidates, X_train, y_train, X_test, y_test
)
print(results)

                    train_MAE  test_MAE  train_RMSE  test_RMSE  train_R2  \
model                                                                      
Naive: mean             2.599     2.490       3.819      3.401     0.000   
Naive: persistence      2.558     4.214       4.068      6.234    -0.135   
Linear Regression       2.243     2.748       3.415      4.259     0.200   
Random Forest           0.958     3.175       1.657      4.151     0.812   
XGBoost (baseline)      0.797     3.577       1.090      4.524     0.918   

                    test_R2  
model                        
Naive: mean          -0.050  
Naive: persistence   -2.529  
Linear Regression    -0.647  
Random Forest        -0.565  
XGBoost (baseline)   -0.858  


## Interpretation

**The naive mean predictor has the lowest test MAE (2.49) of all five models.**
No machine learning model beats it on held-out data.

Observations:

1. **All ML models overfit.** Random Forest and XGBoost achieve very high
   training R² (0.81 and 0.92) but strongly negative test R² (−0.57 and −0.86).
   The train-test gap widens as model complexity increases.

2. **Persistence is the worst model.** Test R² = −2.53 means "copy last year's
   value" is far worse than predicting the mean. This is consistent with
   **mean reversion** in GDP growth: high-growth years tend to be followed
   by lower-growth years.

3. **Linear Regression is the best ML model**, but still does not beat
   the naive mean baseline (test MAE 2.75 vs 2.49; test R² −0.65 vs −0.05).

4. **The ranking is consistent across metrics.** Test MAE, RMSE and R² all
   agree on the ordering of models — this is not a fluke of any single metric.

5. **The train-test gap widens with model capacity.**
   - Linear Regression: train R² 0.20 → test R² −0.65 (gap = 0.85)
   - Random Forest: train R² 0.81 → test R² −0.57 (gap = 1.38)
   - XGBoost baseline: train R² 0.92 → test R² −0.86 (gap = 1.78)

   More complex models fit training data better but generalize worse — the
   classic overfitting signature. This motivates the tuning strategy in
   `05_hyperparameter_tuning.ipynb`.

### Note on the naive mean's test R²

The naive mean baseline predicts the *training* mean for every test row.
Its test R² is slightly negative (−0.05) because the test-period mean
differs from the training-period mean. This is expected — a "perfect"
mean predictor has R² = 0 only when train and test means are identical.

## Preliminary conclusion

On this dataset, next-year GDP growth across EU-27 countries is not
predictable from World Bank indicators at a level that beats a simple
mean forecast. This motivates the hyperparameter tuning that follows
in `05_hyperparameter_tuning.ipynb` — the goal being not to force a win,
but to *understand how tuning affects overfitting*.

In [14]:
results.to_csv(config.PROCESSED_DIR / "baseline_results.csv")
print("Saved.")

Saved.


> **Bottom line**: The naive mean baseline (test MAE 2.49) beats all ML models.
> This notebook establishes that finding; notebook 05 tests whether tuning
> changes it.

## Next steps

- `05_hyperparameter_tuning.ipynb`: Random Search → Grid Search → Manual tuning
  on XGBoost, evaluated with `TimeSeriesSplit` cross-validation on the training
  set only. The test set stays untouched until the very end.
- `06_evaluation_and_findings.ipynb`: final evaluation, feature importance,
  performance by GDP-per-capita group, and country-level error analysis.